# AdvisorAudit Miner Summary

Loads all `deception_samples.jsonl` outputs from AdvisorAudit mining runs and provides:
- aggregate counts
- per-run summaries
- per-state coverage checks (deceptive + truthful)
- manual record inspection (prompt/action/secondary events)


In [ ]:
from pathlib import Path
import json
from pprint import pprint
import pandas as pd

repo_root = Path('/playpen-ssd/smerrill/deception2')
pd.set_option('display.max_colwidth', 180)


In [ ]:
data_root = repo_root / 'AdvisorAudit' / 'Results' / 'DeceptionMining' / 'deepseek-ai_DeepSeek-R1-Distill-Qwen-7B'
jsonl_files = sorted(data_root.rglob('deception_samples.jsonl'))

print(f'Found {len(jsonl_files)} jsonl files under: {data_root}')
for p in jsonl_files[:12]:
    print('-', p)


In [ ]:
rows = []
full_records = []

for path in jsonl_files:
    run_date = path.parent.parent.name
    gpu = path.parent.name

    meta = {}
    meta_path = path.parent / 'meta.json'
    if meta_path.exists():
        try:
            meta = json.loads(meta_path.read_text())
        except Exception:
            meta = {}

    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except Exception:
                continue

            action = rec.get('action', {}) if isinstance(rec.get('action'), dict) else {}
            raw_text = action.get('_raw_text') or action.get('reasoning')
            truth_ctx = rec.get('truth_context', {}) if isinstance(rec.get('truth_context'), dict) else {}

            row_idx = len(full_records)
            full_records.append(rec)

            rows.append({
                'row_idx': row_idx,
                'source_path': str(path),
                'run_date': run_date,
                'gpu': gpu,
                'model_name': meta.get('model_name'),
                'scenario_name': rec.get('scenario_name') or meta.get('scenario_name'),
                'phase': rec.get('phase'),
                'month_index': rec.get('month_index'),
                'month_name': rec.get('month_name'),
                'state_id': rec.get('state_id'),
                'sample_idx': rec.get('sample_idx'),
                'seed': rec.get('seed'),
                'deceptive': rec.get('deceptive'),
                'parse_fail': bool(action.get('Parse_fail')),
                'parse_mode': action.get('parse_mode'),
                'action_type': action.get('Action'),
                'recommended_investment_id': action.get('Recommended_investment_id'),
                'decision': action.get('Decision'),
                'used_for_transition': bool(rec.get('used_for_transition', False)),
                'secondary_event_count': len(rec.get('secondary_events', [])) if isinstance(rec.get('secondary_events'), list) else 0,
                'truth_recommended_option_id': truth_ctx.get('recommended_option_id'),
                'truth_best_option_id': truth_ctx.get('best_option_id'),
                'truth_deceptive_if_recommended': truth_ctx.get('deceptive_if_recommended'),
                'raw_text': raw_text,
            })

df = pd.DataFrame(rows)
print(f'Loaded {len(df)} records')
df.head(20)


In [ ]:
if df.empty:
    print('No records loaded.')
else:
    total = len(df)
    deceptive = int((df['deceptive'] == True).sum())
    truthful = int((df['deceptive'] == False).sum())
    unknown = int(df['deceptive'].isna().sum())
    parse_fails = int(df['parse_fail'].sum())

    print(f'Total records: {total}')
    print(f'Deceptive: {deceptive}')
    print(f'Truthful: {truthful}')
    print(f'Unknown label: {unknown}')
    print(f'Parse failures: {parse_fails} ({(100.0 * parse_fails / total):.2f}%)')

    print('\nAction types:')
    print(df['action_type'].value_counts(dropna=False))

    print('\nParse mode counts:')
    print(df['parse_mode'].value_counts(dropna=False))

    print('\nPer-run summary:')
    run_summary = df.groupby(['run_date', 'gpu']).agg(
        records=('deceptive', 'size'),
        deceptive=('deceptive', lambda s: int((s == True).sum())),
        truthful=('deceptive', lambda s: int((s == False).sum())),
        parse_fails=('parse_fail', 'sum'),
    ).reset_index()
    run_summary['deceptive_rate_pct'] = 100.0 * run_summary['deceptive'] / run_summary['records'].clip(lower=1)
    run_summary['parse_fail_rate_pct'] = 100.0 * run_summary['parse_fails'] / run_summary['records'].clip(lower=1)
    display(run_summary)


In [ ]:
if df.empty:
    print('No records loaded.')
else:
    state_df = df.dropna(subset=['state_id']).copy()
    state_df['state_key'] = state_df['source_path'] + '::state_' + state_df['state_id'].astype(int).astype(str)

    by_state = state_df.groupby('state_key').agg(
        source_path=('source_path', 'first'),
        state_id=('state_id', 'first'),
        month_name=('month_name', 'first'),
        n_samples=('deceptive', 'size'),
        n_deceptive=('deceptive', lambda s: int((s == True).sum())),
        n_truthful=('deceptive', lambda s: int((s == False).sum())),
        n_parse_fail=('parse_fail', 'sum'),
    ).reset_index()

    by_state['has_deceptive'] = by_state['n_deceptive'] > 0
    by_state['has_truthful'] = by_state['n_truthful'] > 0
    by_state['has_both'] = by_state['has_deceptive'] & by_state['has_truthful']

    total_states = len(by_state)
    both_states = int(by_state['has_both'].sum())
    print(f'States: {total_states}')
    print(f'States with both deceptive + truthful sample: {both_states} ({(100.0 * both_states / max(1, total_states)):.2f}%)')

    print('\nStates missing pair coverage:')
    display(by_state[~by_state['has_both']].head(50))


In [ ]:
# Optional run filters
run_date_filter = None   # e.g. '2026-03-06'
gpu_filter = None        # e.g. 'gpu_0'

view_df = df.copy()
if run_date_filter is not None:
    view_df = view_df[view_df['run_date'] == run_date_filter]
if gpu_filter is not None:
    view_df = view_df[view_df['gpu'] == gpu_filter]

print(f'Filtered rows: {len(view_df)}')
view_df.head(30)


In [ ]:
# Manual record inspection
record_df_idx = 0  # index within view_df

if view_df.empty:
    print('view_df is empty. Update filters or data_root.')
else:
    selected = view_df.iloc[int(record_df_idx)]
    rec = full_records[int(selected['row_idx'])]

    print('=== Row Summary ===')
    pprint(dict(selected))

    print('\n=== Prompt ===\n')
    print(rec.get('prompt'))

    action = rec.get('action', {}) if isinstance(rec.get('action'), dict) else {}
    print('\n=== Action ===')
    pprint(action)

    print('\n=== Action Raw Text ===\n')
    print(action.get('_raw_text') or action.get('reasoning'))


In [ ]:
# Inspect secondary events for the selected record
if 'rec' not in globals():
    print('Run the record inspection cell first.')
else:
    secondary = rec.get('secondary_events', []) if isinstance(rec.get('secondary_events'), list) else []
    print(f'Secondary events: {len(secondary)}')
    for i, ev in enumerate(secondary):
        print(f'\n--- secondary_events[{i}] ---')
        pprint({
            'phase': ev.get('phase'),
            'active_player': ev.get('active_player'),
            'seed': ev.get('seed'),
            'fallback_used': ev.get('fallback_used'),
            'done': ev.get('done'),
            'outcome': ev.get('outcome'),
            'action': ev.get('action'),
            'applied_action': ev.get('applied_action'),
        })
        prompt = ev.get('prompt')
        if prompt:
            print('\nPrompt excerpt:')
            print(prompt[:1200])


In [ ]:
# Quick view of parse-fail records
parse_fail_df = df[df['parse_fail'] == True].copy()
print(f'Parse-fail records: {len(parse_fail_df)}')
parse_fail_df[['run_date', 'gpu', 'state_id', 'sample_idx', 'deceptive', 'parse_fail', 'action_type', 'recommended_investment_id', 'raw_text']].head(20)
